In [ ]:
import json
import pandas as pd

# ===== 파일 로드 =====
# json_path = "/workspace/데이터셋/label/train/train_anger.json"
# json_path = "/workspace/데이터셋/label/train/train_happy.json"
# json_path = "/workspace/데이터셋/label/train/train_panic.json"
json_path = "/workspace/데이터셋/label/train/train_sadness.json"

csv_path = "/workspace/user1/data_classification/3차 프로젝트(4조)-오분류 감정 라벨링_수정판.csv"

# Try 'cp949' encoding if 'utf-8' fails for the JSON file
with open(json_path, "r", encoding="cp949") as f:
    data = json.load(f)

df = pd.read_csv(csv_path)
df = df.iloc[:,0:5]


In [79]:
df

,파일명,trn/val,이전 감정,정정 감정,BOX 이상
0,d65g96eca067f324c4a9d01dced7d0016a96856b574c70...,trn,sadness,sadness,Yes
1,9h5g1da6785445a3360aba30bae4454a021dfe8f1f6406...,val,happy,happy,Yes
2,2iy7195d2defabe1b1761e8e148d1fe15d6960b8ddf87d...,trn,sadness,sadness,Yes
3,mbezd31b2d54ff4cc87290207fbe33ee4a7afe5344a788...,trn,happy,happy,Yes
4,0xah1dfff2988dd0bb1002299e49a2e81221721df20867...,trn,sadness,sadness,Yes
...,...,...,...,...,...
2114,vmi7fef28ef05e34f34d825fff8344d8879c88383e7d9b...,trn,anger,panic,No
2115,n7jlc06aa011e1601774ff620c4c7411f7c130bc670ee0...,trn,anger,neutral,No
2116,rsfd568f8c608f3f1e7ae1a13a8e8788a1f02783ae1932...,trn,anger,sadness,No
2117,z17b5a1ff8a235cec1a1aaf585a8d63dc1a043d314ffd8...,trn,anger,sadness,No


In [80]:
data

[{'filename': 'xs3z4f15972eb0ebbb36f3e04654f4f5720614bfd8a5ba48231105a61ca6abmrz.jpg',
  'gender': '여',
  'age': 30,
  'isProf': '전문인',
  'faceExp_uploader': '슬픔',
  'bg_uploader': '문화재 및 유적지',
  'annot_A': {'boxes': {'maxX': 1582.139798564338,
    'maxY': 1530.6656109168403,
    'minX': 1059.6397382777686,
    'minY': 770.1475364515805},
   'faceExp': '슬픔',
   'bg': '문화재 및 유적지'},
  'annot_B': {'boxes': {'maxX': 1590.944141963225,
    'maxY': 1536.4190259647685,
    'minX': 1058.431567127683,
    'minY': 735.8697194897765},
   'faceExp': '기쁨',
   'bg': '실외 자연환경'},
  'annot_C': {'boxes': {'maxX': 1582.1160490782222,
    'maxY': 1530.5828999999997,
    'minX': 1045.1687509217786,
    'minY': 744.6512999999998},
   'faceExp': '중립',
   'bg': '실외 자연환경'}},
 {'filename': 'j3lrc2f227a917cd03e9177b2ce7da237e5d2a91a6e1210c8293922d202d5dw8x.jpg',
  'gender': '남',
  'age': 20,
  'isProf': '일반인',
  'faceExp_uploader': '슬픔',
  'bg_uploader': '숙박 및 거주공간',
  'annot_A': {'boxes': {'maxX': 1959.15970025

In [81]:
import json

# json_path = "train_anger.json"
# json_path = "train_happy.json"
# json_path = "train_panic.json"
json_path = "train_sadness.json"

with open(json_path, "r", encoding="cp949") as f:
    data = json.load(f)

kor_to_eng = {
    "행복": "happiness",
    "분노": "anger",
    "슬픔": "sadness",
    "당황": "panic"
}

change_count = 0

for item in data:
    label = item.get("faceExp_uploader", "").strip()

    if label in kor_to_eng:
        item["faceExp_uploader"] = kor_to_eng[label]
        change_count += 1

# with open("train_anger_eng.json", "w", encoding="cp949") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)
# with open("train_happy_eng.json", "w", encoding="cp949") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)
# with open("train_panic_eng.json", "w", encoding="cp949") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)
with open("train_sadness_eng.json", "w", encoding="cp949") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"변환 완료: {change_count}개")

변환 완료: 1500개


In [82]:
# ===== 매핑 생성 =====
# (file_name, before_label) → after_label
mapping = {}
for _, row in df.iterrows():
    key = (row["파일명"], str(row["이전 감정"]).strip())
    mapping[key] = str(row["정정 감정"]).strip()

# ===== 수정 =====
change_count = 0

for item in data:
    filename = item["filename"]
    current_label = str(item.get("faceExp_uploader", "")).strip()

    key = (filename, current_label)

    if key in mapping:
        item["faceExp_uploader"] = mapping[key]
        change_count += 1

# ===== 저장 =====
# with open("train_anger_modified.json", "w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)
# with open("train_happy_modified.json", "w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)
# with open("train_panic_modified.json", "w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)
with open("train_sadness_modified.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"총 변경 개수: {change_count}")

총 변경 개수: 4
